In [ ]:
! pip install tensorflow nltk datasets numpy

In [ ]:
import tensorflow as tf
from keras.preprocessing.text import Tokenizer
from datasets import load_dataset
from keras.preprocessing.sequence import pad_sequences
import nltk
import numpy as np
from keras.layers import Embedding,Dense,LSTM,Dropout

In [ ]:
nltk.download("punkt")

In [ ]:
dataset = load_dataset("ag_news")
train_data = dataset["train"]
test_data = dataset['test']

In [ ]:
train_texts = [item['text'] for item in train_data]
train_labels = [item['label'] for item in train_data]
test_texts = [item['text'] for item in test_data]
test_labels = [item['label'] for item in test_data]

In [ ]:
vacob = 20000
max_seq_length = 100

tokenizer = Tokenizer(num_words=vacob,oov_token="<UNK>")

tokenizer.fit_on_texts(train_texts)

In [ ]:
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

In [ ]:
train_padded = pad_sequences(sequences=train_sequences,maxlen=max_seq_length,padding="post",truncating="post")
test_padded = pad_sequences(sequences=test_sequences,maxlen=max_seq_length,padding="post",truncating="post")


In [ ]:
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

In [ ]:
Embedding_dim = 128

num_classes = 4 

model = tf.keras.Sequential()
model.add(Embedding(input_dim=vacob,output_dim=Embedding_dim,input_length=max_seq_length))
model.add(LSTM(units=256,return_sequences=False))
model.add(Dropout(0.3))
model.add(Dense(units=num_classes,activation="softmax"))

In [ ]:
model.compile(loss="sparse_categorical",optimizer="adam",metrics='accuracy')
model.summary()

In [ ]:
model.fit(train_padded,train_labels,epochs=5,batch_size = 32,validation_data=(test_padded,test_labels))

In [ ]:
loss,accuracy = model.evaluate(test_padded,test_labels)

print(f"final Accuracy:{accuracy} and loss:{loss}")

In [ ]:
def predict(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence,max_len=max_seq_length,padding="post",truncating='post')
    
    pred = model.predict(padded)
    pred_index = np.argmax(pred,axis=1)[0]
    labels = [
        "World",
        "Sports",
        "Business",
        "Sci/Tech"
    ]
    
    return labels[pred_index]

In [ ]:
print(predict("The stock market saw a huge drop today due to inflation concerns"))